# 96-well Bradford-based Protein Quantification (BPQ)

This notebook owns deck setup and `lh.setup()`. The protocol logic lives in `bpq_protocol.py`, so you can edit the `.py` file and rerun the import + protocol cells without rebuilding the deck each time.

This notebook is standalone. It rebuilds the APE deck context, then adds the BPQ carrier, assay plates, Bradford reagent trough, Bradford standards plate, and the extra 50 uL tip racks needed for sample and standards transfers.

Protein samples from the elution plate are duplicated. Standards are triplicated with a full column serving as blanks.

Three Corning 360 uL flat-bottom plates test all 96 samples from the ElutionPlate with standards and blanks on each plate.

Plate1
col1, col2 = ElutionPlate `A1:H1` samples and duplicates
...
col7, col8 = ElutionPlate `A4:H4` samples and duplicates
col9 = blanks
col10, 11, 12 = standards

Plate2
col1, col2 = ElutionPlate `A5:H5` samples and duplicates
...
col7, col8 = ElutionPlate `A8:H8` samples and duplicates
col9 = blanks
col10, 11, 12 = standards

Plate3
col1, col2 = ElutionPlate `A9:H9` samples and duplicates
...
col7, col8 = ElutionPlate `A12:H12` samples and duplicates
col9 = blanks
col10, 11, 12 = standards

Bradford Standards and Well Locations as listed below:

| Deepwell well | Final BSA concentration |
| ------------- | ------------------------|
| A1 |  2,000 ug/mL |
| B1 |  1,500 ug/mL |
| C1 |  1,000 ug/mL |
| D1 |  750 ug/mL |
| E1 |  500 ug/mL |
| F1 |  250 ug/mL |
| G1 |  125 ug/mL |
| H1 |  25 ug/mL |
| H2 |  0 ug/mL |

## Do This Before Running

- Put the `TIP_CAR_480_A00` at rails 25 with three 1000 uL tip racks in positions `[0]`, `[1]`, and `[2]`.
- Add 50 uL filtered tip racks to `tip_car[3]` and `tip_car[4]`.
- Ensure the APE resources are loaded exactly as described in `APE_96w.ipynb`.
- Insert a new carrier at rails 7 with `Hamilton_MFX_plateholder_DWP_metal_tapped` modules.
- Rails 7, pos `[0]`: `Cor_96_wellplate_360ul_Fb` as `SamplePlate1`.
- Rails 7, pos `[1]`: `Cor_96_wellplate_360ul_Fb` as `SamplePlate2`.
- Rails 7, pos `[2]`: `Cor_96_wellplate_360ul_Fb` as `SamplePlate3`.
- Rails 7, pos `[3]`: `AGenBio_1_troughplate_100000uL_Fl` as `BradfordReagent`.
- Rails 7, pos `[4]`: `BioER_96_wellplate_Vb_2200uL` as `BradfordStds`.


In [1]:
try:
    %load_ext autoreload
    %autoreload 2
except Exception:
    pass

import importlib

from pylabrobot.liquid_handling import LiquidHandler
from pylabrobot.liquid_handling.backends import STARBackend
from pylabrobot.resources.hamilton import STARLetDeck, MFX_CAR_L5_base, TIP_CAR_480_A00
from pylabrobot.resources.hamilton.mfx_modules import Hamilton_MFX_plateholder_DWP_metal_tapped
from pylabrobot.resources.diy.grindbio.modules import Hamilton_MFX_plateholder_DWP_metal_tapped_10mm_3dprint
from pylabrobot.resources.alpaqua import Alpaqua_96_magnum_flx
from pylabrobot.resources.bioer.plates import BioER_96_wellplate_Vb_2200uL
from pylabrobot.resources.agenbio.plates import AGenBio_1_troughplate_100000uL_Fl
from pylabrobot.resources.corning.plates import Cor_96_wellplate_360ul_Fb
from pylabrobot.resources import hamilton_96_tiprack_1000uL, hamilton_96_tiprack_50uL_filter


In [2]:
# Deck layout for the standalone BPQ workflow.
backend = STARBackend()
lh: LiquidHandler = LiquidHandler(backend=backend, deck=STARLetDeck())

deck = STARLetDeck(
  core_grippers="1000uL-at-waste"  # or "1000uL-5mL-on-waste"
)

tip_car = TIP_CAR_480_A00("tip_car")
lh.deck.assign_child_resource(tip_car, rails=25)
tiprack_1000_1 = hamilton_96_tiprack_1000uL("tips_00")
tiprack_1000_2 = hamilton_96_tiprack_1000uL("tips_01")
tiprack_1000_3 = hamilton_96_tiprack_1000uL("tips_02")
tiprack_50_samples = hamilton_96_tiprack_50uL_filter("tips_03")
tiprack_50_standards = hamilton_96_tiprack_50uL_filter("tips_04")
tip_car[0] = tiprack_1000_1
tip_car[1] = tiprack_1000_2
tip_car[2] = tiprack_1000_3
tip_car[3] = tiprack_50_samples
tip_car[4] = tiprack_50_standards

# Rails 13: mag plate + wash / waste / elution resources on 10 mm supports.
rail13_modules = {
    0: Hamilton_MFX_plateholder_DWP_metal_tapped_10mm_3dprint("rail13_mag_module"),
    1: Hamilton_MFX_plateholder_DWP_metal_tapped_10mm_3dprint("rail13_waste_module"),
    2: Hamilton_MFX_plateholder_DWP_metal_tapped_10mm_3dprint("rail13_elution_plate_module"),
    3: Hamilton_MFX_plateholder_DWP_metal_tapped_10mm_3dprint("rail13_wash1_module"),
    4: Hamilton_MFX_plateholder_DWP_metal_tapped_10mm_3dprint("rail13_wash2_module"),
}
car_13 = MFX_CAR_L5_base("car_13", modules=rail13_modules)
lh.deck.assign_child_resource(car_13, rails=13)

mag_plate = Alpaqua_96_magnum_flx("mag_plate")
waste_trough = AGenBio_1_troughplate_100000uL_Fl("waste_trough")
elution_plate = BioER_96_wellplate_Vb_2200uL("elution_plate")
wash1_trough = AGenBio_1_troughplate_100000uL_Fl("wash1_trough")
wash2_trough = AGenBio_1_troughplate_100000uL_Fl("wash2_trough")

rail13_modules[0].assign_child_resource(mag_plate)
rail13_modules[1].assign_child_resource(waste_trough)
rail13_modules[2].assign_child_resource(elution_plate)
rail13_modules[3].assign_child_resource(wash1_trough)
rail13_modules[4].assign_child_resource(wash2_trough)

# Rails 19: binding plate + source / flowthrough / binding buffer / elution buffer.
rail19_modules = {
    0: Hamilton_MFX_plateholder_DWP_metal_tapped("rail19_binding_module"),
    1: Hamilton_MFX_plateholder_DWP_metal_tapped("rail19_source_module"),
    2: Hamilton_MFX_plateholder_DWP_metal_tapped("rail19_flowthrough_module"),
    3: Hamilton_MFX_plateholder_DWP_metal_tapped("rail19_binding_trough_module"),
    4: Hamilton_MFX_plateholder_DWP_metal_tapped("rail19_elution_trough_module"),
}
car_19 = MFX_CAR_L5_base("car_19", modules=rail19_modules)
lh.deck.assign_child_resource(car_19, rails=19)

binding_plate = BioER_96_wellplate_Vb_2200uL("binding_plate")
source_plate = BioER_96_wellplate_Vb_2200uL("source_plate")
flowthrough_plate = BioER_96_wellplate_Vb_2200uL("flowthrough_plate")
binding_trough = AGenBio_1_troughplate_100000uL_Fl("binding_trough")
elution_trough = AGenBio_1_troughplate_100000uL_Fl("elution_trough")

rail19_modules[0].assign_child_resource(binding_plate)
rail19_modules[1].assign_child_resource(source_plate)
rail19_modules[2].assign_child_resource(flowthrough_plate)
rail19_modules[3].assign_child_resource(binding_trough)
rail19_modules[4].assign_child_resource(elution_trough)

# Rails 7: BPQ assay plates + Bradford reagent + Bradford standards.
rail7_modules = {
    0: Hamilton_MFX_plateholder_DWP_metal_tapped("rail7_sample_plate_1_module"),
    1: Hamilton_MFX_plateholder_DWP_metal_tapped("rail7_sample_plate_2_module"),
    2: Hamilton_MFX_plateholder_DWP_metal_tapped("rail7_sample_plate_3_module"),
    3: Hamilton_MFX_plateholder_DWP_metal_tapped("rail7_bradford_reagent_module"),
    4: Hamilton_MFX_plateholder_DWP_metal_tapped("rail7_bradford_stds_module"),
}
car_7 = MFX_CAR_L5_base("car_7", modules=rail7_modules)
lh.deck.assign_child_resource(car_7, rails=7)

sample_plate_1 = Cor_96_wellplate_360ul_Fb("sample_plate_1")
sample_plate_2 = Cor_96_wellplate_360ul_Fb("sample_plate_2")
sample_plate_3 = Cor_96_wellplate_360ul_Fb("sample_plate_3")
bradford_reagent = AGenBio_1_troughplate_100000uL_Fl("bradford_reagent")
bradford_stds = BioER_96_wellplate_Vb_2200uL("bradford_stds")

rail7_modules[0].assign_child_resource(sample_plate_1)
rail7_modules[1].assign_child_resource(sample_plate_2)
rail7_modules[2].assign_child_resource(sample_plate_3)
rail7_modules[3].assign_child_resource(bradford_reagent)
rail7_modules[4].assign_child_resource(bradford_stds)


/tmp/ipykernel_1062886/1947231178.py:30: DeprecationWarning: MFX_CAR_L5_base is deprecated. Use 'hamilton_mfx_carrier_L5_base' instead.
  car_13 = MFX_CAR_L5_base("car_13", modules=rail13_modules)
/tmp/ipykernel_1062886/1947231178.py:33: DeprecationWarning: Alpaqua_96_magnum_flx is deprecated. Use 'alpaqua_96_plateadapter_magnum_flx' instead.
  mag_plate = Alpaqua_96_magnum_flx("mag_plate")
/tmp/ipykernel_1062886/1947231178.py:47: DeprecationWarning: Hamilton_MFX_plateholder_DWP_metal_tapped is deprecated. Use 'hamilton_mfx_plateholder_DWP_metal_tapped' instead.
  0: Hamilton_MFX_plateholder_DWP_metal_tapped("rail19_binding_module"),
/tmp/ipykernel_1062886/1947231178.py:48: DeprecationWarning: Hamilton_MFX_plateholder_DWP_metal_tapped is deprecated. Use 'hamilton_mfx_plateholder_DWP_metal_tapped' instead.
  1: Hamilton_MFX_plateholder_DWP_metal_tapped("rail19_source_module"),
/tmp/ipykernel_1062886/1947231178.py:49: DeprecationWarning: Hamilton_MFX_plateholder_DWP_metal_tapped is depre

In [3]:
await lh.setup(skip_autoload=True)

# STARlet without iSWAP reports 0 arms; keep Co-Re gripper bookkeeping available.
if lh.backend.num_arms == 0:
    lh._resource_pickups = {0: None}

print(lh.deck.get_resource("core_grippers"))


2026-04-09 17:54:02,008 - pylabrobot.io.usb - INFO - Finding USB device...
2026-04-09 17:54:02,019 - pylabrobot.io.usb - INFO - Found USB device.
2026-04-09 17:54:02,022 - pylabrobot.io.usb - INFO - Found endpoints. 
Write:
       ENDPOINT 0x2: Bulk OUT ===============================
       bLength          :    0x7 (7 bytes)
       bDescriptorType  :    0x5 Endpoint
       bEndpointAddress :    0x2 OUT
       bmAttributes     :    0x2 Bulk
       wMaxPacketSize   :   0x40 (64 bytes)
       bInterval        :    0x0 
Read:
       ENDPOINT 0x81: Bulk IN ===============================
       bLength          :    0x7 (7 bytes)
       bDescriptorType  :    0x5 Endpoint
       bEndpointAddress :   0x81 IN
       bmAttributes     :    0x2 Bulk
       wMaxPacketSize   :   0x40 (64 bytes)
       bInterval        :    0x0
2026-04-09 17:54:05,191 - pylabrobot - INFO - Running backend initialization procedure.


HamiltonCoreGrippers(name='core_grippers', location=Coordinate(022.500, -29.500, 105.000), size_x=39, size_y=61, size_z=24, category=core_grippers)


In [4]:
# PLR deck setup overview.
print(lh.summary())


Rail  Resource                      Type                 Coordinates (mm)
(-6)  ├── trash_core96              Trash                (-58.200, 106.000, 216.400)
      │
(7)   ├── car_7                     MFXCarrier           (235.000, 063.000, 100.000)
      │   ├── sample_plate_1        Plate                (239.000, 072.000, 180.920)
      │   ├── sample_plate_2        Plate                (239.000, 168.000, 180.920)
      │   ├── sample_plate_3        Plate                (239.000, 264.000, 180.920)
      │   ├── bradford_reagent      Plate                (239.000, 360.000, 179.210)
      │   ├── bradford_stds         Plate                (239.000, 456.000, 179.210)
      │
(13)  ├── car_13                    MFXCarrier           (370.000, 063.000, 100.000)
      │   ├── mag_plate             PlateAdapter         (374.000, 072.000, 138.195)
      │   ├── waste_trough          Plate                (374.000, 168.000, 133.455)
      │   ├── elution_plate         Plate                (37

In [5]:
import bpq_protocol

bpq_protocol = importlib.reload(bpq_protocol)


In [10]:
await bpq_protocol.run_bpq_96w(
    lh,
    elution_plate=elution_plate,
    sample_plate_1=sample_plate_1,
    sample_plate_2=sample_plate_2,
    sample_plate_3=sample_plate_3,
    bradford_reagent=bradford_reagent,
    bradford_stds=bradford_stds,
    tiprack_1000=tiprack_1000_1,
    sample_tiprack_50=tiprack_50_samples,
    standards_tiprack_50=tiprack_50_standards,
    add_reagent_sample_plate_1=False,
    add_samples_plate_1=True,
    add_stds_plate_1=True,
    add_reagent_sample_plate_2=False,
    add_samples_plate_2=True,
    add_stds_plate_2=True,
    add_reagent_sample_plate_3=True,
    add_samples_plate_3=True,
    add_stds_plate_3=True,
    reagent_volume=250,
    reagent_aspirate_volume=800,
    sample_aspirate_volume=12,
    sample_dispense_volume=5,
    standard_aspirate_volume=18,
    standard_dispense_volume=5,
)


Starting BPQ 96-well protocol.
Bradford reagent: plate 1/1 (SamplePlate3), columns 1-3.
Bradford reagent: plate 1/1 (SamplePlate3), columns 4-6.
Bradford reagent: plate 1/1 (SamplePlate3), columns 7-9.
Bradford reagent: plate 1/1 (SamplePlate3), columns 10-12.
Skipping Bradford reagent addition for SamplePlate1.
Elution column 1 duplicated to assay columns 1 and 2.
Elution column 2 duplicated to assay columns 3 and 4.
Elution column 3 duplicated to assay columns 5 and 6.
Elution column 4 duplicated to assay columns 7 and 8.
Completed elution sample transfer for SamplePlate1.
Bradford standards added to assay columns 10-12 using tip column 1.
Completed Bradford standards transfer for SamplePlate1.
Skipping Bradford reagent addition for SamplePlate2.
Elution column 5 duplicated to assay columns 1 and 2.
Elution column 6 duplicated to assay columns 3 and 4.
Elution column 7 duplicated to assay columns 5 and 6.
Elution column 8 duplicated to assay columns 7 and 8.
Completed elution sample 

In [ ]:
lh.summary()


In [ ]:
# await lh.dispense(
#                 bradford_reagent["A1"]*8,
#                 vols=[100]*8,
#                 use_channels=[0,1,2,3,4,5,6,7],
#                 liquid_height = [4]*8, # the perfect height for 1000ul. About 1mm when finished. 
#                 flow_rates=[200]*8,
#                 # auto_surface_following_distance=True,
#                 blow_out=[1]*8, 
#                 swap_speed=[160]*8,
#                 settling_time=[1]*8
#             )

In [7]:
# await lh.drop_tips(tiprack_1000_1["A1:H1"], use_channels=[0,1,2,3,4,5,6,7])
# await lh.discard_tips()
# 